In [1]:
import pandas as pd
import numpy as np

# ── MODEL: The "Logic Engine" ──────────────────────────────────────────
class DiabetesModel:
    def __init__(self, input_filepath):
        self.filepath = input_filepath
        self.df = None
        self.df_clean = None
        self.cleaning_stats = {}

    def load_data(self):
        self.df = pd.read_csv(self.filepath)
        self.df_clean = self.df.copy()
        return self.df

    def process_data(self):
        # Columns where 0 is actually a biological impossibility (missing data)
        zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
        
        # 1. Replace 0 with NaN (Requires numpy)
        for col in zero_cols:
            self.df_clean[col] = self.df_clean[col].replace(0, np.nan)
        
        # 2. Record counts for the View to display
        self.cleaning_stats['null_counts'] = self.df_clean[zero_cols].isnull().sum().to_dict()
        
        # 3. Impute: Fill NaNs with Median
        for col in zero_cols:
            median_val = self.df_clean[col].median()
            self.df_clean[col] = self.df_clean[col].fillna(median_val)
            self.cleaning_stats[f'{col}_median'] = median_val
            
        return self.df_clean

    def save_data(self, output_filepath):
        self.df_clean.to_csv(output_filepath, index=False)

# ── VIEW: The "Presentation Layer" ─────────────────────────────────────
class ConsoleView:
    def show_raw_stats(self, df):
        print("\n" + "="*40)
        print("  DIABETES PIPELINE: DATA LOADED")
        print("="*40)
        print(f"Total Rows: {len(df)}\n")
        print(df.head())

    def show_cleaning_summary(self, stats, df_clean):
        print("\n" + "="*40)
        print("  CLEANING SUMMARY (0 -> MEDIAN)")
        print("="*40)
        for col, count in stats['null_counts'].items():
            print(f"  - {col:15}: {count} values fixed (Median: {stats[f'{col}_median']:.2f})")
        
        print("\nCleaned Statistics (Partial):")
        print(df_clean[['Glucose', 'Insulin', 'BMI']].describe().loc[['mean', 'max']])

    def show_save_success(self, path):
        print(f"\n[SUCCESS]: Processed data saved to {path}")

# ── CONTROLLER: The "Orchestrator" ─────────────────────────────────────
class PipelineController:
    def __init__(self, input_file, output_file):
        self.model = DiabetesModel(input_file)
        self.view = ConsoleView()
        self.output_file = output_file

    def run(self):
        # Step 1: Load
        raw_data = self.model.load_data()
        self.view.show_raw_stats(raw_data)
        
        # Step 2: Process
        self.model.process_data()
        self.view.show_cleaning_summary(self.model.cleaning_stats, self.model.df_clean)
        
        # Step 3: Save
        self.model.save_data(self.output_file)
        self.view.show_save_success(self.output_file)

# ── EXECUTION ──────────────────────────────────────────────────────────
if __name__ == "__main__":
    # Ensure diabetes.csv is in the same folder
    controller = PipelineController('diabetes.csv', 'diabetes_cleaned.csv')
    controller.run()


  DIABETES PIPELINE: DATA LOADED
Total Rows: 768

   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  

  CLEANING SUMMARY (0 -> MEDIAN)
  - Glucose        : 5 values fixed (Median: 117.00)
  - BloodPressure  : 35 values fixed (Median: 72.00)
  - SkinThickness  : 227 values fixed (Median: 29.00)
  - Insulin        : 374 values fixed (Median: 125.00)
  -